# Prepare for DB

The following code, keeps only the non duplicated nodes and excludes the "fga" entries.
Analyzing the code I noticed that allActsPDF.csv has the same entries as allActs.csv

In [ ]:
import re

# Function to clean the content
def clean_content(content):
    cleaned = []
    buffer = ""

    for line in content:
        line = line.strip()
        
        # Check if the line starts with a URL or is a continuation
        if line.startswith('"https://') or buffer == "":
            # If there's something in the buffer, append it first
            if buffer:
                cleaned.append(buffer.rstrip(","))  # Remove trailing comma before appending
            buffer = re.sub(r'\s*,\s*', ',', line)  # Remove whitespaces around commas
        else:
            # Line continuation case
            buffer += " " + line

    # Append the last buffer if exists
    if buffer:
        cleaned.append(buffer.rstrip(","))
    
    return cleaned

# Clean the content
cleaned_content = clean_content(content)

# Count the fixed lines
split_lines = len(content) - len(cleaned_content)

split_lines, cleaned_content[:20]  # Show the first 20 cleaned lines and the number of fixed lines


In [14]:
import pandas as pd

try:
    # Attempt to read the fixed CSV file
    df = pd.read_csv('./raw_data/DBpopulation.csv')
    print(f"Successfully read CSV with {len(df)} rows and {len(df.columns)} columns")
    print("Column names:", df.columns.tolist())
    print("\nFirst 3 rows:")
    print(df.head(3).to_string())
    
    # Check for any missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\nMissing values by column:")
        print(missing[missing > 0])
    else:
        print("\nNo missing values found.")
        
except Exception as e:
    print(f"Error reading CSV: {e}")

Successfully read CSV with 46254 rows and 7 columns
Column names: ['act', 'title', 'publicationDate', 'decisionDate', 'entryintoforce', 'fileUrl', 'typeDoc']

First 3 rows:
                                                       act                                                                                                                                 title publicationDate decisionDate entryintoforce                                                                                                                                                       fileUrl                       typeDoc
0  https://fedlex.data.admin.ch/eli/oc/1998/2833_2833_2833                                                   Ordinanza sui dazi per le merci dall'AELS e dalle CE (Ordinanza sul libero scambio)      1998-12-15   1998-10-23     1998-11-01  https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/oc/1998/2833_2833_2833/it/pdf-a/fedlex-data-admin-ch-eli-oc-1998-2833_2833_2833-it-pdf-a.pdf  Ordina

In [ ]:
# find which rows have missing values in publication_date
df = pd.read_csv('./raw_data/DBpopulation.csv')
df[7318:]

,act,title,publicationDate,decisionDate,entryintoforce,fileUrl,typeDoc
7317,https://fedlex.data.admin.ch/eli/oc/2022/614,Scambio di note del 7 ottobre 2022 tra la Sviz...,2022-10-26,2022-10-07,2022-10-07,https://fedlex.data.admin.ch/filestore/fedlex....,Trattato internazionale bilaterale
7318,https://fedlex.data.admin.ch/eli/oc/2022/686,Protocollo tra la Svizzera e il Giappone che m...,2022-11-17,2021-07-16,2022-11-30,https://fedlex.data.admin.ch/filestore/fedlex....,Trattato internazionale bilaterale
7319,https://fedlex.data.admin.ch/eli/oc/2023/12,Accordo quadro del 20 settembre 2022 tra il Co...,2023-01-13,2022-09-20,2022-12-29,https://fedlex.data.admin.ch/filestore/fedlex....,Trattato internazionale bilaterale
7320,https://fedlex.data.admin.ch/eli/oc/2013/674,Protocollo che modifica la Convenzione tra la ...,2013-11-05,2012-06-25,2013-10-21,https://fedlex.data.admin.ch/filestore/fedlex....,Trattato internazionale bilaterale
7321,https://fedlex.data.admin.ch/eli/oc/2020/74,Scambio di note del 28/30 gennaio 2020 tra la ...,2020-02-11,2020-01-30,2020-02-01,https://fedlex.data.admin.ch/filestore/fedlex....,Trattato internazionale bilaterale
...,...,...,...,...,...,...,...
46249,https://fedlex.data.admin.ch/eli/oc/2025/135,Legge federale concernente l’imposta sul valor...,2025-02-27,2025-02-19,2025-02-27,https://fedlex.data.admin.ch/filestore/fedlex....,Correzione RU
46250,https://fedlex.data.admin.ch/eli/oc/2025/136,Convenzione internazionale del 18 maggio 2007 ...,2025-02-27,2025-02-25,2025-02-25,https://fedlex.data.admin.ch/filestore/fedlex....,Campo d'applicazione
46251,https://fedlex.data.admin.ch/eli/oc/2025/137,Convenzione n. 142 del 23 giugno 1975 concerne...,2025-02-27,2025-02-24,2025-02-24,https://fedlex.data.admin.ch/filestore/fedlex....,Campo d'applicazione
46252,https://fedlex.data.admin.ch/eli/oc/2025/138,Ordinanza sull’energia (OEn),2025-02-27,2025-02-19,2026-01-01,https://fedlex.data.admin.ch/filestore/fedlex....,Ordinanza del Consiglio federale


### Clean the dataset

In [10]:
import pandas as pd
import re

# Load the cleaned CSV file
file_path = './raw_data/old_DBpopulation_2.csv'  # Replace with your file path
df = pd.read_csv(file_path)

# Function to clean the quotes and whitespace
def clean_value(value):
    if isinstance(value, str):
        value = value.replace('"', '')  # Remove quotes
        value = re.sub(r'\s*,\s*', ',', value)  # Trim whitespace around commas
        return value.strip()  # Remove leading and trailing whitespace
    return value

# Apply cleaning function to all columns except 'title'
for col in df.columns:
    if col != 'title':
        df[col] = df[col].apply(clean_value)

# Save the cleaned DataFrame
output_file_path = './raw_data/DBpopulation.csv'
df.to_csv(output_file_path, index=False)

print(f"Cleaned file saved to: {output_file_path}")



Cleaned file saved to: ./raw_data/DBpopulation.csv


In [15]:
import pandas as pd

# Load the cleaned CSV file
file_path = './raw_data/DBpopulation.csv'  # Make sure the file path is correct
df = pd.read_csv(file_path)

# Check for duplicate rows
duplicate_rows = df[df.duplicated()]
print(f"Number of duplicate rows: {len(duplicate_rows)}")

if not duplicate_rows.empty:
    print("Duplicate rows:")
    print(duplicate_rows)

# Check for duplicated values in the 'act' column
if 'act' in df.columns:
    duplicate_acts = df[df.duplicated(subset='act', keep=False)]
    print(f"\nNumber of duplicated 'act' values: {len(duplicate_acts)}")

    if not duplicate_acts.empty:
        print("Duplicated 'act' rows:")
        print(duplicate_acts.head())
else:
    print("\nColumn 'act' not found in the dataset.")


Number of duplicate rows: 0

Number of duplicated 'act' values: 0


In [5]:
import pandas as pd

# Load the cleaned CSV file
file_path = './raw_data/queryResults_cleaned.csv'  # Replace with your file path
df = pd.read_csv(file_path)

# Drop rows where 'act' column contains "fga" or "cc" (case-insensitive)
df_filtered = df[~df['act'].str.contains(r'\b(fga|cc)\b', case=False, na=False)]

# Save the filtered DataFrame to a new CSV file
output_file_path = './raw_data/queryResults_cleaned_filtered.csv'  # Desired output file name
df_filtered.to_csv(output_file_path, index=False)

print(f"Filtered CSV file saved to: {output_file_path}")


/tmp/ipykernel_467243/1087115237.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_filtered = df[~df['act'].str.contains(r'\b(fga|cc)\b', case=False, na=False)]


Filtered CSV file saved to: ./raw_data/queryResults_cleaned_filtered.csv


In [ ]:
df = pd.read_csv('./raw_data/queryResults_cleaned_filtered.csv')
df.to

# Populate the DB


In [ ]:
from graphdatascience import GraphDataScience
user_pass_neo4j = ("neo4j", "chlawskg")
driver = GraphDataScience("bolt://localhost:49873", auth=user_pass_neo4j)

driver.run_cypher("""CREATE (l:Law) SET l.name = $id SET l.type = $type""", {"id": "1", "type": "law"})

## list of attributes
- year | number -> dateId
- act.value -> pageId
- title.value -> title
- publicationDate.value -> publicationDate
- entryintoforce.value -> entryintoforceDate
- typeDoc.value -> typeLaw

In [ ]:
cypher_query = """CREATE (l:Law) SET l.dateId = $dateId 
                                 SET l.pageId = $pageId 
                                 SET l.title = $title 
                                 SET l.publicationDate = date($publicationDate) 
                                 SET l.entryintoforceDate = date($entryintoforceDate) 
                                 SET l.typeLaw = $typeLaw"""
params = {"dateId": "dateId", "pageId": "act.value", "title": "title.value", "publicationDate": "publicationDate.value", "entryintoforceDate": "entryintoforce.value", "typeLaw": "typeDoc.value"}
driver.run_cypher(cypher_query, params)

In [2]:
import pandas as pd

db_population = pd.read_csv('./raw_data/DBpopulation.csv')
db_population

,act.type,act.value,title.type,title.value,publicationDate.type,publicationDate.value,entryintoforce.type,entryintoforce.value,typeDoc.type,typeDoc.xml:lang,typeDoc.value,fileUrl.type,fileUrl.value
0,uri,https://fedlex.data.admin.ch/eli/oc/1/82_69_80,literal,Trattato d'estradizione tra la Svizzera e l'Im...,literal,1874-06-03,literal,1874-06-03,literal,it,Trattato internazionale bilaterale,NaN,NaN
1,uri,https://fedlex.data.admin.ch/eli/oc/18/654_588...,literal,Ordinanza concernente la consegna e il control...,literal,1901-04-19,literal,1901-04-19,literal,it,Ordinanza del Consiglio federale,NaN,NaN
2,uri,https://fedlex.data.admin.ch/eli/oc/18/865_796...,literal,Ordinanza del 9 novembre 1901 sull'esercizio d...,literal,1901-11-09,literal,1901-11-09,literal,it,Ordinanza del Consiglio federale,NaN,NaN
3,uri,https://fedlex.data.admin.ch/eli/oc/18/870_801...,literal,Ordinanza sull'esercizio della Fabbrica federa...,literal,1901-11-09,literal,1901-11-09,literal,it,Ordinanza del Consiglio federale,NaN,NaN
4,uri,https://fedlex.data.admin.ch/eli/oc/19/816_779...,literal,Decreto del Consiglio federale del 24 dicembre...,literal,1906-07-01,literal,1906-07-01,literal,it,Ordinanza del Consiglio federale,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
46125,uri,https://fedlex.data.admin.ch/eli/oc/2024/688,literal,Ordinanza concernente gli effettivi massimi pe...,literal,2024-11-25,literal,2025-01-01,literal,it,Ordinanza del Consiglio federale,uri,https://fedlex.data.admin.ch/filestore/fedlex....
46126,uri,https://fedlex.data.admin.ch/eli/oc/2024/692,literal,Ordinanza concernente l’imposizione minima dei...,literal,2024-11-26,literal,2025-01-01,literal,it,Ordinanza del Consiglio federale,uri,https://fedlex.data.admin.ch/filestore/fedlex....
46127,uri,https://fedlex.data.admin.ch/eli/oc/2024/693,literal,Ordinanza sulla costituzione di una riserva di...,literal,2024-11-26,literal,2025-01-01,literal,it,Ordinanza del Consiglio federale,uri,https://fedlex.data.admin.ch/filestore/fedlex....
46128,uri,https://fedlex.data.admin.ch/eli/oc/2024/697,literal,Ordinanza sull’assicurazione malattie (OAMal),literal,2024-11-26,literal,2025-01-01,literal,it,Ordinanza del Consiglio federale,uri,https://fedlex.data.admin.ch/filestore/fedlex....


In [ ]:
# for every row in the dataframe, create a node in the graph
for index, row in db_population.iterrows():
    cypher_query = """CREATE (l:Law) SET l.dateId = $dateId 
                                     SET l.pageId = $pageId 
                                     SET l.title = $title 
                                     SET l.publicationDate = date($publicationDate) 
                                     SET l.entryintoforceDate = date($entryintoforceDate) 
                                     SET l.typeLaw = $typeLaw"""
    dateId = row['dateId']
    params = {"dateId": dateId, "pageId": row['act.value'], "title": row['title.value'], "publicationDate": row['publicationDate.value'], "entryintoforceDate": row['entryintoforce.value'], "typeLaw": row['typeDoc.value']}
    driver.run_cypher(cypher_query, params)

In [ ]:
import os
import urllib.request
import pandas as pd
from requests_html import HTMLSession

session = HTMLSession()

for index, row in db_population.iterrows():
    try:
        page_url = row['act.value']
        # open url and save the code
        response = session.get(page_url)
        response.html.render(sleep=2)
        label = response.html.find("app-memorial-label", first=True)
        if label:
            text = label.text
            code = text.split()[-2:]
        else:
            with open("output_log.txt", "a") as log_file:
                log_file.write(f"No code found for index {index}\n")
            if(row['publicationDate.value'][6:] == '1998'):
                code = row['act.value'].split('/')[-1].split('_')[0]
            else:
                code = row['act.value'].split('/')[-1]
        dateId = code[0] + "|" + code[1]
        cypher_query = """CREATE (l:Law) SET l.dateId = $dateId 
                                     SET l.pageId = $pageId 
                                     SET l.title = $title 
                                     SET l.publicationDate = date($publicationDate) 
                                     SET l.entryintoforceDate = date($entryintoforceDate) 
                                     SET l.typeLaw = $typeLaw"""
        params = {"dateId": dateId, "pageId": row['act.value'], "title": row['title.value'], "publicationDate": row['publicationDate.value'], "entryintoforceDate": row['entryintoforce.value'], "typeLaw": row['typeDoc.value']}
        driver.run_cypher(cypher_query, params)

    except Exception as e:
        print(e)
        print(f"Error at index {index}")
        continue

    break

session.close()

In [ ]:

from requests_html import HTMLSession
import pandas as pd

page_url = "https://fedlex.data.admin.ch/eli/oc/2000/150"
session = HTMLSession()


try:
    response = session.get(page_url)
    response.html.render(sleep=2)
    label = response.html.find("app-memorial-label", first=True)
    if label:
        text = label.text
        code = text.split()[-2:]
    else:
        print("No code found")
    
    dateId = code[0] + "|" + code[1]

except Exception as e:
    print(e)

session.close()

Index where pattern first changes: 1413
Instances of pattern after index 1413: 23316
Examples of similar URLs where the pattern reappears:
['https://fedlex.data.admin.ch/eli/oc/1983/1080_1080_1080', 'https://fedlex.data.admin.ch/eli/oc/43/479_499_495', 'https://fedlex.data.admin.ch/eli/oc/44/305_316_312', 'https://fedlex.data.admin.ch/eli/oc/44/808_866_854', 'https://fedlex.data.admin.ch/eli/oc/45/15_15_15']


In [7]:
# check if the db_population dataframe has some entries that match with double_number_pattern = re.compile(r"^\d+_\d+$")
import re
from numpy import single
import pandas as pd
db_population = pd.read_csv('./raw_data/DBpopulation.csv')

single_number_pattern = re.compile(r"^\d+$")
double_number_pattern = re.compile(r"^\d+_\d+$")
triple_number_pattern = re.compile(r"^\d+_\d+_\d+$")
count = 0
for index, row in db_population.iterrows():
    if single_number_pattern.match(row['act.value'].split('/')[-1]):
        count += 1

print(count)

21083


## Let's check which rows from the db are not present in the neo4j db

In [ ]:
from graphdatascience import GraphDataScience
import pandas as pd

db_population = pd.read_csv('./raw_data/DBpopulation.csv')
user_pass_neo4j = ("neo4j", "chlawskg")
driver = GraphDataScience("bolt://localhost:49873", auth=user_pass_neo4j)

cypher_query = """MATCH (l:Law) WHERE l.pageId ENDS WITH $pageId RETURN l"""
missing_laws = []

for index, row in db_population.iterrows():
    pageId = row['act.value']
    params = {"pageId": pageId}
    query_result = driver.run_cypher(cypher_query, params)
    if query_result.empty:
        missing_laws.append(pageId)
        print(f"Missing law: {pageId}")
    

missing_laws_df = pd.DataFrame(missing_laws, columns=['act.value'])
#missing_laws_df.to_csv('./raw_data/missing_laws.csv', index=False)

driver.close()

# Remove laws that don't follow the format to simplify the process for now

In [18]:
import pandas as pd
import re
db = pd.read_csv('./raw_data/DBpopulation_params.csv')

# only keep the duplicates that respect the following formats in the last element obtained from splitting pageId on "/" : either one number or three numbers separated by "_"
single_number_pattern = re.compile(r"^\d+$")
triple_number_pattern = re.compile(r"^\d+_\d+_\d+$")

# Extract the last element from pageId URL path and check if it matches either pattern
filtered_db = db[db['pageId'].apply(
    lambda x: bool(single_number_pattern.match(x.split('/')[-1]) or 
                   triple_number_pattern.match(x.split('/')[-1]))
)]

# find all rows with lawId equal to RU 0 0
zero_db = filtered_db[filtered_db['lawId'] == 'RU 0 0']

# remove all zero_db rows from the filtered_db
filtered_db = filtered_db[filtered_db['lawId'] != 'RU 0 0']
filtered_db.to_csv('./raw_data/DBpopulation_final.csv', index=False)

In [20]:
# Create a new DataFrame with just the columns you want to check for duplicates
id_cols = ['gerId', 'frId', 'itId']

# Find duplicates - this returns True for each row that's a duplicate of a previous row
duplicates = filtered_db.duplicated(subset='gerId', keep='first')

# Get the duplicate rows
duplicate_rows = filtered_db[duplicates]

# Display the duplicates, if any
print(f"Found {len(duplicate_rows)} duplicate sets of (gerId, frId, itId)")
if len(duplicate_rows) > 0:
    print(duplicate_rows)
    
# To see all rows that have duplicates (including the first occurrence)
all_duplicates = filtered_db[filtered_db.duplicated(subset=id_cols, keep=False)]
if len(all_duplicates) > 0:
    print("\nAll rows involved in duplicates (grouped):")
    print(all_duplicates.sort_values(by=id_cols))

Found 29 duplicate sets of (gerId, frId, itId)
              lawId         gerId          frId          itId  \
9035   RU 1972 2454  AS 1972 2620  RO 1972 2675  RU 1972 2454   
19298  RU 1971 1958  AS 1971 1952  RO 1971 1959  RU 1971 1958   
19383   RU 1957 192   AS 1957 179   RO 1957 183   RU 1957 192   
21543   RU 1966 864   AS 1966 846   RO 1966 872   RU 1966 864   
22401   RU 1973 225   AS 1973 223   RO 1973 223   RU 1973 225   
25212     RU 16 904     AS 16 885     RO 16 827     RU 16 904   
25236       RU 25 6       AS 25 6       RO 25 6       RU 25 6   
26142  RU 1948 1106  AS 1948 1144  RO 1948 1132  RU 1948 1106   
26400   RU 1964 931   AS 1964 915   RO 1964 910   RU 1964 931   
27298    RU VII 183    AS VII 183    RO VII 187    RU VII 183   
28026   RU 1968 172   AS 1968 163   RO 1968 170   RU 1968 172   
28188   RU 1951 532   AS 1951 528   RO 1951 526   RU 1951 532   
29135  RU 1951 1188  AS 1951 1156  RO 1951 1160  RU 1951 1188   
32224  RU 1971 1957  AS 1971 1951  RO 1971 

In [4]:
import re

duplicates = pd.read_csv('./raw_data/DBpopulation_params.csv')

# only keep the duplicates that respect the following formats in the last element obtained from splitting pageId on "/" : either one number or three numbers separated by "_"
single_number_pattern = re.compile(r"^\d+$")
triple_number_pattern = re.compile(r"^\d+_\d+_\d+$")

# Extract the last element from pageId URL path and check if it matches either pattern
filtered_duplicates = duplicates[duplicates['pageId'].apply(
    lambda x: bool(single_number_pattern.match(x.split('/')[-1]) or 
                  triple_number_pattern.match(x.split('/')[-1]))
)]

# only show duplicated gerId, frId, itId
filtered_duplicates = filtered_duplicates[filtered_duplicates.duplicated(subset=['gerId', 'frId', 'itId'], keep=False)]

# Display the filtered duplicates
filtered_duplicates

# save as csv
#filtered_duplicates.to_csv('./raw_data/broken_codes.csv', index=False)

,lawId,gerId,frId,itId,pageId,title,publicationDate,decisionDate,entryintoforceDate,typeLaw
14136,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2015/668,Ordinanza sull'impiego da parte di autorità fe...,2015-10-20,2015-10-20,2015-10-20,Correzione RU
15631,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2018/779,Ordinanza del DATEC concernente le indicazioni...,2018-12-18,2018-11-23,2019-01-01,Ordinanza di un dipartimento
36194,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2014/209,Ordinanza sul coordinamento dei controlli dell...,2014-04-23,2014-04-23,2014-04-23,Correzione RU
39960,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2025/23,Ordinanza dell’USAV sulla detenzione di animal...,2025-01-14,2024-12-20,2025-02-01,Ordinanza di un'ufficio
40122,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2009/761,Ordinanza concernente la perequazione finanzia...,2009-12-08,2009-11-18,2010-01-01,Ordinanza del Consiglio federale
41054,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2017/483,Convenzione europea del 13 novembre 1987 per l...,2017-08-22,2017-08-10,2017-08-10,Campo d'applicazione
41195,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2016/308,Ordinanza dell'Ufficio federale delle comunica...,2016-06-07,2016-05-26,2016-06-13,Ordinanza di un'ufficio
41220,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2020/275,Ordinanza 2 sui provvedimenti per combattere i...,2020-04-29,2020-04-29,2020-04-24,Ordinanza del Consiglio federale
41379,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2018/223,Ordinanza sul collocamento e il personale a pr...,2018-04-17,2018-04-17,2018-07-01,Correzione RU
41658,RU 0 0,AS 0 0,RO 0 0,RU 0 0,https://fedlex.data.admin.ch/eli/oc/2004/37,Ordinanza del DFI sui requisiti igienici-micro...,2004-01-27,2003-12-15,2004-02-01,Ordinanza di un dipartimento


## now let's merge the two dbs into the final one

In [25]:
import pandas as pd

db_population_zero = pd.read_csv('./raw_data/Dbpopulation_zero.csv')
db_population_final = pd.read_csv('./raw_data/DBpopulation_final.csv')

# Add rows from db_population_zero to db_population_final
db_population_final = pd.concat([db_population_final, db_population_zero])
db_population_final.to_csv('./raw_data/DBpopulation_neo4j.csv', index=False)